In [ ]:
# Install pyspark and pyarrow cleanly
!pip install -q pyspark pyarrow

In [ ]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, pandas_udf
from pyspark.sql.types import DoubleType

# This will now initialize without throwing the Java gateway error
spark = SparkSession.builder.master("local[*]").appName("ColabPandasUDF").getOrCreate()

# Create dummy mock data for our tutorial
data = [
    (1, "A", 10.0, 2.0),
    (1, "B", 20.0, 4.0),
    (2, "A", 30.0, 6.0),
    (2, "B", 40.0, 8.0),
]
schema = ["group_id", "category", "value_x", "value_y"]
df = spark.createDataFrame(data, schema=schema)

print("Spark Session Successfully Started!")
df.show()

Spark Session Successfully Started!
+--------+--------+-------+-------+
|group_id|category|value_x|value_y|
+--------+--------+-------+-------+
|       1|       A|   10.0|    2.0|
|       1|       B|   20.0|    4.0|
|       2|       A|   30.0|    6.0|
|       2|       B|   40.0|    8.0|
+--------+--------+-------+-------+



Series to Series UDF

In [ ]:
# Type hint specifies that inputs and outputs are Pandas Series
@pandas_udf(DoubleType())
def scale_values(x: pd.Series, y: pd.Series) -> pd.Series:
    # This runs as a fast, vectorized Pandas operation under the hood
    return (x * 2) + y


# Apply to the DataFrame
df_scaled = df.withColumn("scaled_result", scale_values(col("value_x"), col("value_y")))
df_scaled.show()

+--------+--------+-------+-------+-------------+
|group_id|category|value_x|value_y|scaled_result|
+--------+--------+-------+-------+-------------+
|       1|       A|   10.0|    2.0|         22.0|
|       1|       B|   20.0|    4.0|         44.0|
|       2|       A|   30.0|    6.0|         66.0|
|       2|       B|   40.0|    8.0|         88.0|
+--------+--------+-------+-------+-------------+



Iterator of Series to Iterator of Series

In [ ]:
from typing import Iterator


@pandas_udf(DoubleType())
def batch_multiplier(iterator: Iterator[pd.Series]) -> Iterator[pd.Series]:
    # 1. Setup Phase: This happens ONCE when the partition loads
    # Imagine loading a heavy pickle file or model weight here
    coefficient = 1.5

    # 2. Execution Phase: Process the stream of data batches natively
    for s in iterator:
        yield s * coefficient


# Apply to the DataFrame
df_multiplied = df.withColumn("multiplied", batch_multiplier(col("value_x")))
df_multiplied.show()

+--------+--------+-------+-------+----------+
|group_id|category|value_x|value_y|multiplied|
+--------+--------+-------+-------+----------+
|       1|       A|   10.0|    2.0|      15.0|
|       1|       B|   20.0|    4.0|      30.0|
|       2|       A|   30.0|    6.0|      45.0|
|       2|       B|   40.0|    8.0|      60.0|
+--------+--------+-------+-------+----------+



Series to Scalar

In [ ]:
@pandas_udf(DoubleType())
def custom_mean_diff(x: pd.Series, y: pd.Series) -> float:
    # Returns a single scalar float per group
    return float(x.mean() - y.mean())


# Apply using groupBy
df_agg = df.groupBy("group_id").agg(
    custom_mean_diff(col("value_x"), col("value_y")).alias("mean_diff")
)
df_agg.show()

+--------+---------+
|group_id|mean_diff|
+--------+---------+
|       1|     12.0|
|       2|     28.0|
+--------+---------+

